# 🤖 13장 · 몸을 가진 AI: 센서 데이터로 읽는 피지컬 AI (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 13장의 **05. 파이썬 실습**과 **종합 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 13-1, 13-2, 13-3, 13-4, 13-5, 13-6 |
| ☆ 선택 | 자습이나 심화 | 13-7, 13-8, 13-9, 13-10, 13-11 |

**데이터**
- 동작 인식: UCI "Human Activity Recognition Using Smartphones" (30명, 2.56초 윈도우 10,299개). GitHub 저장소 `bidhanbaray/UCI_HAR_Dataset`에서 필요한 파일만 내려받습니다(약 200MB).
- 공장 기계: UCI "AI4I 2020 Predictive Maintenance Dataset" (작업 10,000건, 합성 데이터). GitHub 저장소 `lingchenlijing/AI4I-2020-Predictive-Maintenance-Dataset`에서 바로 읽습니다.
- 종합 프로젝트: 한국 공공데이터 파일을 직접 올려서 씁니다(맨 아래 안내).

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch13/data/"
for f in [
    "asos_서울108_시간자료_2023_2026.csv",
    "kpx_시간별_전국_전력수요량_2021_2025.csv",
    "서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv",
    "에어코리아_서울_도시대기평균_시간별_2023_2026.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 허리의 휴대폰에서 공장의 기계까지

이 절의 실습은 03절의 분석을 순서대로 다시 합니다. 두 데이터 모두 회원 가입 없이 받을 수 있는 공개 데이터이고, 이 책에서는 깃허브에 올라 있는 사본을 씁니다. 동작 인식 데이터의 원신호 파일은 모두 합쳐 약 200MB라 처음 내려받을 때 시간이 조금 걸립니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 13-1 ★] 동작 인식 데이터 내려받고 누가 만들었는지 보기

HAR 데이터의 원신호 9개 채널(몸 가속도, 각속도, 전체 가속도의 세 축)을 (윈도우 수, 시간 128칸, 채널 9) 모양의 배열로 불러오시오. 훈련과 시험에 나온 사람, 사람마다 윈도우 수, 동작별 윈도우 수를 구하시오.

① 필요한 파일을 내려받습니다. ② 채널마다 표를 읽어 3차원 배열로 쌓습니다. ③ 사람 번호와 동작을 셉니다.

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

HAR = "UCI HAR Dataset"
MIRROR = "https://raw.githubusercontent.com/bidhanbaray/UCI_HAR_Dataset/master/"
CH = [f"{s}_{a}" for s in ["body_acc", "body_gyro", "total_acc"] for a in "xyz"]

def get_har():                             # 필요한 파일만 깃허브 사본에서 받는다
    files = []
    for s in ["train", "test"]:
        files += [f"{s}/y_{s}.txt", f"{s}/subject_{s}.txt"]
        files += [f"{s}/Inertial Signals/{c}_{s}.txt" for c in CH]
    for f in files:
        if not os.path.exists(f"{HAR}/{f}"):
            os.makedirs(os.path.dirname(f"{HAR}/{f}"), exist_ok=True)
            urllib.request.urlretrieve(MIRROR + f.replace(" ", "%20"), f"{HAR}/{f}")

def load(split):                           # (윈도우 수, 시간 128칸, 채널 9)
    sig = np.stack([np.loadtxt(f"{HAR}/{split}/Inertial Signals/{c}_{split}.txt")
                    for c in CH], axis=2)
    y = np.loadtxt(f"{HAR}/{split}/y_{split}.txt").astype(int) - 1
    s = np.loadtxt(f"{HAR}/{split}/subject_{split}.txt").astype(int)
    return sig, y, s

get_har()
sig_tr, y_tr, s_tr = load("train")
sig_te, y_te, s_te = load("test")
print("훈련:", sig_tr.shape, " 시험:", sig_te.shape)
print("한 윈도우 =", sig_tr.shape[1] / 50, "초 (50Hz)")

ACT = ["걷기", "계단 오르기", "계단 내려가기", "앉기", "서기", "눕기"]
y_all, s_all = np.r_[y_tr, y_te], np.r_[s_tr, s_te]
print("훈련에 나온 사람:", sorted(map(int, set(s_tr))))
print("시험에 나온 사람:", sorted(map(int, set(s_te))))
per = pd.Series(s_all).value_counts().sort_index()
print("사람 수:", len(per), " 한 사람의 윈도우 수:", per.min(), "~", per.max())
print(pd.Series(y_all).map(dict(enumerate(ACT))).value_counts().reindex(ACT).to_string())

color = ["#C8234B" if s in set(s_te) else "#9a9a9a" for s in per.index]
per.plot.bar(color=color, figsize=(7.5, 3))
plt.xlabel("사람 번호 (빨강: 시험용 9명, 회색: 훈련용 21명)")
plt.ylabel("윈도우 수")
plt.xticks(rotation=0, fontsize=10)
plt.show()

훈련 7,352개, 시험 2,947개 윈도우입니다. 정답은 1~6으로 적혀 있어 1을 빼 0~5로 맞췄습니다. 연구팀이 사람 단위로 나누어 두어서, 훈련과 시험에 겹치는 사람이 없습니다. 나이와 성별 칸은 없습니다.

> **[코드 읽기] `np.stack( ..., axis=2)`**
>
> `np.loadtxt( )`는 채널 하나를 (윈도우 수, 128) 표로 읽습니다. `np.stack( )`은 같은 모양의 표 9장을 겹쳐 3차원 배열을 만듭니다. `axis=2`는 "세 번째 방향으로 쌓아라"는 뜻이라, 결과의 모양이 (윈도우 수, 128, 9)가 됩니다. 2장에서 본 `.shape`로 늘 모양을 확인해 보십시오.

#### [파이썬 실습 13-2 ★] 중력이 알려 주는 자세, FFT가 알려 주는 리듬

동작마다 전체 가속도의 축별 평균(중력 방향)과 몸 흔들림의 표준편차 중앙값을 구하시오. 그리고 전체 가속도 크기를 FFT해 걷기의 가장 센 주파수와 1분 걸음 수를 구하시오.

① 동작별로 평균을 냅니다. ② (식 13-1)로 크기를 구해 표준편차를 냅니다. ③ FFT로 가장 센 주파수를 찾습니다.

In [ ]:
sig_all = np.r_[sig_tr, sig_te]
grav = pd.DataFrame([sig_all[y_all == i][:, :, 6:9].mean(axis=(0, 1)) for i in range(6)],
                    index=ACT, columns=["x", "y", "z"])
body = np.linalg.norm(sig_all[:, :, 0:3], axis=2).std(axis=1)    # 몸 흔들림의 표준편차
grav["흔들림(중앙값)"] = [np.median(body[y_all == i]) for i in range(6)]
print(grav.round(3))

fr = np.fft.rfftfreq(128, 1 / 50)          # 0, 0.39, 0.78, ... 25Hz
def spec(x):                               # 윈도우마다 주파수별 세기
    x = x - x.mean(axis=1, keepdims=True)
    return np.abs(np.fft.rfft(x * np.hanning(128), axis=1))

mag = np.linalg.norm(sig_all[:, :, 6:9], axis=2)   # 전체 가속도의 크기
dom = fr[1:][spec(mag)[:, 1:].argmax(axis=1)]      # 가장 센 주파수
walk = np.median(dom[y_all == 0])
print("주파수 간격:", round(fr[1], 3), "Hz,  가장 높은 주파수:", fr[-1], "Hz")
print("걷기의 가장 센 주파수(중앙값):", round(walk, 2), "Hz → 1분에", round(walk * 60), "걸음")

grav[["x", "y", "z"]].plot.bar(color=["#C8234B", "#9a9a9a", "#4d4d4d"], figsize=(6.5, 3.2))
plt.xticks(range(6), ["걷기", "오르기", "내려가기", "앉기", "서기", "눕기"], rotation=0)
plt.ylabel("평균 가속도 (g)")
plt.legend(title="축", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

for i, c in [(0, "#C8234B"), (3, "#7a7a7a")]:   # 걷기와 앉기의 평균 스펙트럼
    plt.plot(fr, spec(mag)[y_all == i].mean(axis=0), color=c, label=ACT[i])
plt.xlim(0, 10)
plt.xlabel("주파수 (Hz, 1초에 몇 번 흔들리나)")
plt.ylabel("세기 (평균)")
plt.legend()
plt.gcf().set_size_inches(6, 3)
plt.show()

`sig_all[..., 6:9]`이 전체 가속도의 세 축이고, `mean(axis=(0, 1))`로 윈도우와 시간에 걸쳐 평균을 냈습니다. 눕기만 x축의 평균이 0.069로 떨어집니다. 흔들림은 움직이는 세 동작과 멈춘 세 동작이 20배 넘게 차이 납니다. FFT 전에 평균을 빼는 것은 0Hz 칸(중력)이 가장 세게 나오는 것을 막기 위해서입니다. 그래프는 3.2의 두 그림입니다.

> **[코드 읽기] `np.fft.rfft( )`와 `rfftfreq( )`**
>
> `np.fft.rfft(x, axis=1)`은 윈도우마다(행마다) FFT를 해서 칸 65개의 세기를 돌려줍니다. `np.fft.rfftfreq(128, 1 / 50)`은 그 칸들이 몇 Hz인지(0, 0.39, …, 25)를 알려 줍니다. `np.hanning(128)`을 곱하는 것은 윈도우 양 끝을 부드럽게 줄여, 잘린 자리에서 생기는 가짜 주파수를 줄이기 위해서입니다.

#### [파이썬 실습 13-3 ★] 특징 110개 만들고 작은 나무로 분류하기

신호 11개(9채널과 전체 가속도·각속도의 크기)마다 통계 10개를 구해 특징 110개를 만드시오. 질문 하나짜리 나무로 움직임과 멈춤을 가르고, 깊이를 바꾸며 여섯 동작의 정확도를 구하시오.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def feats(sig):                            # 신호 11개 × 통계 10개 = 특징 110개
    chans = [sig[:, :, j] for j in range(9)] + [np.linalg.norm(sig[:, :, 6:9], axis=2),
                                                np.linalg.norm(sig[:, :, 3:6], axis=2)]
    cols = []
    for c in chans:
        Sp = spec(c)
        band = lambda lo, hi: (Sp[:, (fr >= lo) & (fr < hi)] ** 2).sum(axis=1)
        q75, q25 = np.percentile(c, [75, 25], axis=1)
        cols += [c.mean(1), c.std(1), c.min(1), c.max(1), q75 - q25,
                 np.sqrt((c ** 2).mean(1)), fr[1:][Sp[:, 1:].argmax(1)],
                 band(0.3, 3), band(3, 10), band(10, 25.1)]
    return np.column_stack(cols)

F_tr, F_te = feats(sig_tr), feats(sig_te)
print("특징 표:", F_tr.shape)
j = 9 * 10 + 1                             # 10번째 신호(전체 가속도 크기)의 표준편차
stump = DecisionTreeClassifier(max_depth=1).fit(F_tr[:, [j]], y_tr <= 2)
print("질문 하나: 흔들림 >", round(stump.tree_.threshold[0], 4), "이면 움직이는 중")
print("움직임/멈춤 정확도:", stump.score(F_te[:, [j]], y_te <= 2))
acc_d = {}
for d in [2, 3, 4, 6, 8]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(F_tr, y_tr)
    acc_d[d] = t.score(F_te, y_te)
    print(f"깊이 {d} 나무: 6동작 정확도 {acc_d[d]:.3f}, 마디 {t.tree_.node_count}개")

pd.Series(acc_d).plot(marker="o", color="#C8234B", figsize=(5, 3))
plt.xlabel("나무의 깊이 (질문을 몇 번 이어 하나)")
plt.ylabel("6동작 정확도")
plt.ylim(0.5, 1)
plt.show()

통계 10개는 평균, 표준편차, 최솟값, 최댓값, 사분위 범위, RMS(제곱 평균의 제곱근), 가장 센 주파수, 세 주파수 대역의 세기입니다. `j = 9 * 10 + 1`은 10번째 신호(전체 가속도 크기)의 두 번째 통계(표준편차)입니다. 정답 `y_tr <= 2`는 걷기, 계단 오르기, 계단 내려가기(0~2)이면 True입니다. 깊이 4(마디 17개)에서 깊이 8(마디 141개)로 마디를 8배 넘게 늘려도 3.6%p밖에 오르지 않았습니다.

> **[코드 읽기] `lambda`**
>
> `band = lambda lo, hi: ...`는 이름 없는 작은 함수를 한 줄로 만듭니다. `def band(lo, hi): return ...`와 같습니다. `band(0.3, 3)`이라고 부르면 0.3~3Hz 칸들의 세기를 제곱해 더한 값을 돌려줍니다.

#### [파이썬 실습 13-4 ★] 처음 보는 사람으로 시험하기

랜덤 포레스트로 (1) 사람 단위로 나눈 원래의 시험과 (2) 윈도우를 무작위로 나눈 시험의 정확도를 비교하시오. 시험 9명의 사람별 정확도와 앉기·서기 혼동도 구하시오.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
p_rf = rf.fit(F_tr, y_tr).predict(F_te)
print("처음 보는 9명으로 시험:", round((p_rf == y_te).mean() * 100, 2), "%")

F_all = np.r_[F_tr, F_te]                  # 사람을 섞어서 무작위로 나누면
A, B, ya, yb = train_test_split(F_all, y_all, test_size=len(y_te) / len(y_all),
                                random_state=42, stratify=y_all)
rf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(A, ya)
print("윈도우를 무작위로 나눠 시험:", round(rf2.score(B, yb) * 100, 2), "%")

acc = pd.Series({s: (p_rf[s_te == s] == y_te[s_te == s]).mean() * 100 for s in sorted(set(s_te))})
print("사람별 정확도:", acc.round(1).to_dict())
cm = confusion_matrix(y_te, p_rf)
print("앉기→서기로 착각:", cm[3, 4], "  서기→앉기로 착각:", cm[4, 3])

acc.sort_values().plot.bar(color="#9a9a9a", figsize=(6, 3), rot=0)
plt.axhline((p_rf == y_te).mean() * 100, color="#C8234B", linestyle="--")
plt.xlabel("시험용 사람 번호 (점선: 9명 평균)")
plt.ylabel("정확도 (%)")
plt.ylim(60, 100)
plt.show()

`train_test_split( )`에 사람 정보를 주지 않으면 윈도우를 무작위로 섞습니다. 시험 비율(2,947 ÷ 10,299)과 동작 비율(`stratify`)을 원래 나눔과 같게 맞췄는데도 7.4%p가 높게 나왔습니다. 이웃한 윈도우의 절반이 같은 숫자이기 때문입니다. 혼동행렬 `cm[3, 4]`는 실제 앉기(3)를 서기(4)로 판정한 수입니다.

#### [파이썬 실습 13-5 ★] 사람 수를 늘리면

훈련에 쓰는 사람 수를 1, 3, 6, 12, 21명으로 바꾸어(21명 미만은 무작위로 5번 뽑아 평균) 처음 보는 9명에 대한 정확도를 구하시오. 또 윈도우를 800개로 맞추고 3명과 21명에게서 뽑았을 때를 비교하시오.

In [ ]:
rng = np.random.default_rng(0)
people = np.array(sorted(set(s_tr)))
RF = lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
curve = {}
for k in [1, 3, 6, 12, 21]:                # 훈련에 쓰는 사람 수를 늘리면
    accs = []
    for rep in range(5 if k < 21 else 1):
        m = np.isin(s_tr, rng.choice(people, k, replace=False))
        accs.append((RF().fit(F_tr[m], y_tr[m]).predict(F_te) == y_te).mean() * 100)
    curve[k] = np.mean(accs)
    print(f"{k:2d}명: 평균 {np.mean(accs):.1f}% ({min(accs):.1f}~{max(accs):.1f}%)")
for k in [3, 21]:                          # 윈도우 800개를 3명에게서, 21명에게서
    accs = []
    for rep in range(5):
        idx = np.where(np.isin(s_tr, rng.choice(people, k, replace=False)))[0]
        idx = rng.choice(idx, 800, replace=False)
        accs.append((RF().fit(F_tr[idx], y_tr[idx]).predict(F_te) == y_te).mean() * 100)
    print(f"윈도우 800개, {k}명에게서: {np.mean(accs):.1f}%")

pd.Series(curve).plot(marker="o", color="#C8234B", figsize=(5, 3))
plt.xlabel("훈련에 쓴 사람 수")
plt.ylabel("처음 보는 9명 정확도 (%)")
plt.show()

`np.isin(s_tr, 뽑은 사람들)`은 "이 윈도우의 주인이 뽑은 사람 안에 있는가"를 True와 False로 돌려줍니다. 사람이 1명뿐이면 누구를 뽑느냐에 따라 71.0%에서 82.6%까지 달라집니다. 랜덤 포레스트를 서른 번 넘게 배우므로 코랩에서는 1분 남짓 걸립니다.

#### [파이썬 실습 13-6 ★] 공장 데이터: 너무 깨끗한지 보고 물리 특징 더하기

AI4I 2020 데이터에 온도 차, 출력, 공구 부담 특징을 만들고, 설명서의 고장 규칙이 라벨과 얼마나 맞는지 확인하시오. 그리고 8,000건으로 배우고 2,000건으로 시험해, 센서 값 그대로와 물리 특징을 더한 랜덤 포레스트를 견주시오.

① 데이터를 읽고 특징 셋을 만듭니다. ② 규칙과 라벨을 견줍니다. ③ 두 모델의 정밀도, 재현율, F1을 구합니다. ④ 출력 구간별 고장률을 그립니다.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/lingchenlijing/AI4I-2020-"
                 "Predictive-Maintenance-Dataset/master/data/ai4i2020.csv")
df.columns = [c.replace("﻿", "") for c in df.columns]
print(df.shape, " 고장:", df["Machine failure"].sum(), "건")
df["dT"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60
df["strain"] = df["Tool wear [min]"] * df["Torque [Nm]"]
print(df[["Air temperature [K]", "dT", "Torque [Nm]"]].agg(["mean", "std"]).round(2))
rules = {"HDF": (df.dT < 8.6) & (df["Rotational speed [rpm]"] < 1380),
         "PWF": (df.power < 3500) | (df.power > 9000),
         "OSF": df.strain > df.Type.map({"L": 11000, "M": 12000, "H": 13000})}
for m, rule in rules.items():
    print(f"{m}: 규칙 {rule.sum()}건, 라벨 {df[m].sum()}건, 겹침 {(rule & (df[m] == 1)).sum()}건")

from sklearn.metrics import f1_score, precision_score, recall_score
num = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]",
       "Torque [Nm]", "Tool wear [min]"]
df["TypeN"] = df["Type"].map({"L": 0, "M": 1, "H": 2})
base, phys = num + ["TypeN"], num + ["TypeN", "dT", "power", "strain"]
y = df["Machine failure"].values
i_tr, i_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
print("시험 2,000건 가운데 고장:", y[i_te].sum(), "건, 모두 정상으로 찍으면",
      round((y[i_te] == 0).mean() * 100, 1), "%")
for name, cols in [("센서값 그대로", base), ("물리 특징 추가", phys)]:
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    m.fit(df.loc[i_tr, cols], y[i_tr])
    pr = m.predict_proba(df.loc[i_te, cols])[:, 1]
    p = (pr >= 0.5).astype(int)
    print(f"{name}: 정밀도 {precision_score(y[i_te], p):.3f} 재현율 {recall_score(y[i_te], p):.3f}"
          f" F1 {f1_score(y[i_te], p):.3f}")

pb = pd.cut(df.power, [0, 3500, 5000, 7000, 9000, 20000])
rate = df.groupby(pb, observed=True)["Machine failure"].agg(["size", "mean"])
print(rate.round(3).to_string())
(rate["mean"] * 100).plot.bar(color=["#C8234B", "#9a9a9a", "#9a9a9a", "#9a9a9a", "#C8234B"],
                              figsize=(6, 3), rot=0)
plt.xticks(range(5), ["3,500 미만", "3,500~5,000", "5,000~7,000", "7,000~9,000", "9,000 초과"],
           fontsize=10)
plt.xlabel("기계적 출력 (W)")
plt.ylabel("고장률 (%)")
plt.show()

열 이름 앞의 보이지 않는 글자(파일의 첫머리 표시)는 지웠습니다. 세 규칙이 라벨과 한 건도 어긋나지 않고, 표준편차가 2.0, 1.0처럼 딱 떨어집니다. 합성 데이터의 흔적입니다. `stratify=y`로 두 묶음의 고장 비율을 같게 맞췄습니다. 시험 2,000건을 모두 정상으로 찍어도 96.6%이므로, 정확도 대신 정밀도와 재현율을 봅니다. 그래프는 3.7의 그림입니다.

#### [파이썬 실습 13-7 ☆] 기울기 각도를 특징으로 더하기

(식 13-2)로 중력과 휴대폰 세 축의 각도를 구해 앉기, 서기, 눕기의 평균을 비교하고, 110개 특징에 더했을 때 정확도와 앉기·서기 혼동이 어떻게 바뀌는지 구하시오.

In [ ]:
def angle(sig):                            # 중력과 휴대폰 각 축 사이의 각도
    g = sig[:, :, 6:9].mean(axis=1)
    g = g / np.linalg.norm(g, axis=1, keepdims=True)
    return np.degrees(np.arccos(np.clip(g, -1, 1)))

ang = angle(sig_all)
print(pd.DataFrame([ang[y_all == i].mean(axis=0) for i in (3, 4, 5)],
                   index=ACT[3:], columns=["x축", "y축", "z축"]).round(1))
G_tr, G_te = np.c_[F_tr, angle(sig_tr)], np.c_[F_te, angle(sig_te)]
p_g = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(G_tr, y_tr).predict(G_te)
cm_g = confusion_matrix(y_te, p_g)
print("각도 3개를 더한 정확도:", round((p_g == y_te).mean() * 100, 2), "%")
print("앉기·서기 혼동:", cm[3, 4] + cm[4, 3], "→", cm_g[3, 4] + cm_g[4, 3], "건")

`np.clip(g, -1, 1)`은 반올림 오차로 1을 살짝 넘는 값이 `arccos`에 들어가지 않게 막습니다. 앉기의 x축 각도(16.9°)가 서기(12.9°)보다 크지만, 특징으로 더해도 나아지지 않았습니다. 혼동은 135건에서 147건으로 오히려 늘었습니다.

#### [파이썬 실습 13-8 ☆] 연구팀의 특징 561개와 견주기

연구팀이 만들어 공개한 특징 561개로 로지스틱 회귀를 배워, 우리가 만든 110개와 정확도를 견주시오.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

for s in ["train", "test"]:                # 연구팀이 만든 특징 561개 (약 92MB)
    f = f"{HAR}/{s}/X_{s}.txt"
    if not os.path.exists(f):
        urllib.request.urlretrieve(MIRROR + f"{s}/X_{s}.txt", f)
X561_tr, X561_te = np.loadtxt(f"{HAR}/train/X_train.txt"), np.loadtxt(f"{HAR}/test/X_test.txt")
lr561 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
print("특징 561개 + 로지스틱 회귀:", round(lr561.fit(X561_tr, y_tr).score(X561_te, y_te) * 100, 2), "%")
lr110 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
print("특징 110개 + 로지스틱 회귀:", round(lr110.fit(F_tr, y_tr).score(F_te, y_te) * 100, 2), "%")

연구팀은 특징 561개와 서포트 벡터 머신으로 처음 보는 9명에 대해 약 96%를 보고했습니다. 같은 561개에 로지스틱 회귀를 쓰면 95.59%, 우리의 110개는 91.01%입니다. 특징을 공들여 만들수록 모델은 단순해도 됩니다. 561개 특징 파일은 약 92MB라 내려받는 데 시간이 걸립니다.

#### [파이썬 실습 13-9 ☆] 문턱값은 검증용 데이터로 고르기

고장을 놓치는 손해가 헛경보보다 열 배 크다고 하시오. 훈련 8,000건을 다시 6,000건과 검증용 2,000건으로 나누어 문턱값을 고르고, 그 문턱값을 시험 2,000건에 한 번만 쓰시오.

In [ ]:
cost = lambda yt, p: 10 * ((p == 0) & (yt == 1)).sum() + ((p == 1) & (yt == 0)).sum()
i_fit, i_val = train_test_split(i_tr, test_size=0.25, random_state=0, stratify=y[i_tr])
mv = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
pv = mv.fit(df.loc[i_fit, phys], y[i_fit]).predict_proba(df.loc[i_val, phys])[:, 1]
rows = []
for t in [0.1, 0.2, 0.3, 0.5, 0.7]:        # 문턱값은 훈련 안의 검증용 2,000건으로 고른다
    p = (pv >= t).astype(int)
    rows.append([t, p.sum(), (p & (y[i_val] == 1)).sum(), cost(y[i_val], p)])
val = pd.DataFrame(rows, columns=["문턱값", "경보", "맞힌 고장", "손해(놓침×10+헛경보)"])
print(val.to_string(index=False))
best = val.loc[val.iloc[:, 3].idxmin(), "문턱값"]
p = (pr >= best).astype(int)              # pr: 실습 13-6의 물리 특징 모델이 시험 2,000건에 낸 확률
print(f"고른 문턱값 {best} → 시험: 경보 {p.sum()}건, 맞힌 고장 {(p & (y[i_te] == 1)).sum()}건,"
      f" 재현율 {recall_score(y[i_te], p):.3f}, 손해 {cost(y[i_te], p)}")
p5 = (pr >= 0.5).astype(int)
print(f"문턱값 0.5 그대로 → 시험: 경보 {p5.sum()}건, 손해 {cost(y[i_te], p5)}")

문턱값을 시험 데이터로 고르면 시험 문제를 보고 답을 고친 셈이 됩니다. 그래서 훈련 안에서 검증용을 떼어 골랐습니다. 검증에서는 0.3의 손해가 75로 가장 작았습니다. 시험에 쓰니 고장 68건 가운데 56건을 찾았고, 손해는 0.5를 그대로 쓸 때(132)보다 조금 작은 127이었습니다. 처음 보는 데이터에서는 검증 때보다 덜 맞는 것이 보통입니다.

#### [파이썬 실습 13-10 ☆] 정답 없이 이상한 작업 찾기

고장 라벨을 쓰지 않고 센서 값 5개만으로 고립 숲을 배워 가장 이상한 3.39%를 고르고, 그 가운데 실제 고장의 비율과 고장 유형별로 찾은 수를 구하시오.

In [ ]:
from sklearn.ensemble import IsolationForest

Xn = StandardScaler().fit_transform(df[num])
iso = IsolationForest(n_estimators=300, contamination=0.0339, random_state=42)
flag = iso.fit_predict(Xn) == -1           # 라벨 없이 "이상하다"고 고른 것
fail = df["Machine failure"] == 1
print("이상으로 고른 것:", flag.sum(), "건, 그 가운데 실제 고장:", (flag & fail).sum(), "건")
print("정밀도:", round((flag & fail).sum() / flag.sum() * 100, 1), "%  (전체 고장 비율 3.39%)")
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print({m: f"{(flag & (df[m] == 1)).sum()}/{df[m].sum()}" for m in modes})

현장에서는 고장 기록, 곧 정답이 없을 때가 많습니다. 그럴 때는 동떨어진 작업을 찾는 이상 탐지를 씁니다. 고립 숲은 데이터를 무작위 질문으로 잘게 나누어, 몇 번 만에 혼자 떨어지는지를 셉니다. 고른 339건 가운데 실제 고장은 54건(15.9%)이었습니다. 전체 고장 비율의 4.7배이지만, 정답으로 배운 모델보다는 한참 못합니다. 출력이 극단으로 치우치는 전력 이상(39/95)은 잘 찾았습니다. 값 하나하나는 평범한 열 방출 실패(5/115)는 거의 찾지 못했습니다.

#### [파이썬 실습 13-11 ☆] 모델의 크기 재기

깊이 3 나무, 신경망(64-32), 랜덤 포레스트 200그루의 정확도와 저장 파일 크기를 구하시오. 신경망은 (식 13-6)으로 매개변수 메모리도 구하시오.

In [ ]:
import pickle
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600,
                                                    early_stopping=True, random_state=0))
mlp.fit(F_tr, y_tr)
n_par = sum(w.size for w in mlp[-1].coefs_) + sum(b.size for b in mlp[-1].intercepts_)
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(F_tr, y_tr)
for name, model, n in [("깊이 3 나무", t3, None), ("신경망 64-32", mlp, n_par), ("랜덤 포레스트 200그루", rf, None)]:
    kb = len(pickle.dumps(model)) / 1024
    extra = f", 매개변수 {n:,}개 × 4바이트 = {n * 4 / 1024:.1f}KB" if n else ""
    print(f"{name}: 정확도 {model.score(F_te, y_te) * 100:.2f}%, 저장 파일 {kb:,.1f}KB{extra}")

`pickle.dumps( )`는 모델을 파일로 저장할 때의 바이트를 돌려줍니다. 랜덤 포레스트는 나무 200그루의 마디를 모두 저장해 11MB가 넘습니다. 신경망의 저장 파일이 매개변수 메모리(36.6KB)보다 큰 것은 학습 기록 같은 덧붙은 정보까지 함께 저장되기 때문입니다.

> **핵심 3줄**
>
> 사람 단위로 나눈 시험은 90.23%, 윈도우를 무작위로 섞은 시험은 97.66%입니다. 평균 뒤에는 77.9%에 그친 사람이 숨어 있습니다.
>
> 훈련에 쓴 사람이 1명에서 21명으로 늘면 정확도가 75.1%에서 90.2%로 오릅니다. 같은 윈도우 800개라도 3명보다 21명에게서 뽑을 때 더 높습니다(80.9% → 87.7%).
>
> AI4I는 세 고장 규칙이 라벨과 한 건도 어긋나지 않는 합성 데이터입니다. 온도 차와 출력 같은 물리 특징을 더하면 F1이 0.724에서 0.880으로 오릅니다.

## 04절 계산을 파이썬으로 확인하기

책 04절의 손계산(식 13-1~13-6)을 파이썬으로 다시 해 봅니다.

### 4.1 벡터의 크기와 기울기 각도

In [ ]:
import numpy as np
a = np.array([0, 0.6, 0.8])               # 가만히 놓인 휴대폰의 가속도 (g)
size = np.linalg.norm(a)                   # (식 13-1)
print("크기:", round(size, 3), "g")
print("각도(°):", np.degrees(np.arccos(a / size)).round(1))   # (식 13-2)

### [예제 13-1] 걸음 리듬에서 1분 걸음 수 구하기

In [ ]:
fs, N = 50, 128
freq = np.fft.rfftfreq(N, 1 / fs)          # 칸마다의 주파수
print("가장 높은 주파수:", fs / 2, "Hz,  칸 간격:", fs / N, "Hz")
print("4~6번째 칸:", freq[4:7].round(3), "Hz")
print("5번째 칸 → 1분에", round(freq[5] * 60), "걸음")

### [예제 13-2] 밀링 작업의 출력 구하기

In [ ]:
for torque, rpm in [(40, 1500), (60, 1450)]:
    P = torque * 2 * np.pi * rpm / 60      # (식 13-5)
    print(f"토크 {torque}Nm, {rpm}rpm → {P:,.0f}W")

### 4.4 모델의 크기: 매개변수와 바이트

In [ ]:
layers = [110, 64, 32, 6]                  # 입력 110개 → 64 → 32 → 6동작
n = sum(a * b + b for a, b in zip(layers[:-1], layers[1:]))
print("매개변수:", f"{n:,}", "개")
print("4바이트 실수:", round(n * 4 / 1024, 1), "KB,  1바이트 정수:", round(n / 1024, 1), "KB")

## [미니 프로젝트 13] 종합 프로젝트: 우리 도시의 센서로 질문 하나에 답하기


**질문 고르기.** 서울에는 기상 관측소, 대기 측정소, 지하철 개찰구, 전력 계량기 같은 센서가 흩어져 있습니다. 이들이 남긴 공공데이터로 아래 세 질문 가운데 하나를 골라 답하십시오.

- 주제 A. 기온을 알면 내일 전국의 전력 수요를 얼마나 맞힐 수 있을까? (8장 회귀, 12장 기준선)
- 주제 B. 비가 오는 평일에는 지하철 승객이 줄까? (4장 그림, 5장 검정)
- 주제 C. 오늘의 초미세먼지와 날씨로 내일 "나쁨"을 맞힐 수 있을까? (9장 분류, 10장 나무, 12장 시간 순서)

**데이터.** 모두 공공누리 조건에 따라 출처를 밝히고 쓸 수 있는 한국 공공데이터입니다. 프로젝트에 맞게 줄인 네 파일이 강의 자료에 들어 있습니다. 파일을 코랩 왼쪽 파일 창에 끌어다 놓고 시작합니다. 날씨 파일은 세 주제가 함께 씁니다.

**[표 13-3] 종합 프로젝트 데이터**

| 파일 | 출처 | 크기 | 한 행 |
|---|---|---|---|
| asos_서울108_시간자료_2023_2026.csv | 기상청 종관기상관측(ASOS) 서울 관측소 | 33,241행 × 7열 | 한 시각의 기온, 비, 바람, 습도 |
| kpx_시간별_전국_전력수요량_2021_2025.csv | 한국전력거래소 | 1,827행 × 25열 | 하루, 열은 1~24시의 수요(MW) |
| 서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv | 서울교통공사 | 1,462행 × 22열 | 하루, 승차 또는 하차. 열은 시간대별 인원과 합계(모든 역의 합) |
| 에어코리아_서울_도시대기평균_시간별_2023_2026.csv | 한국환경공단 에어코리아 확정자료 | 29,184행 × 4열 | 한 시각, 서울 도시대기 측정소 25곳의 PM2.5 평균 |

원자료는 코랩에 올리기에 큽니다. 에어코리아 원자료는 서울 측정소 40곳의 시간별 기록 117만 행(약 131MB)이라, 도로변 측정소 15곳을 빼고 도시대기 25곳을 시각마다 평균했습니다. 지하철 원자료는 한 해에 약 20만 행(역마다, 날마다, 승차와 하차)이라, 날짜와 승하차마다 모든 역을 더했습니다. 날씨 파일에는 프로젝트에 쓰는 열만 남겼습니다.

**데이터 파일 올리기.** 프로젝트에 맞게 줄인 아래 네 파일이 강의 자료에 들어 있습니다. 코랩 왼쪽의 📁 파일 창에 끌어다 놓으십시오. 고른 주제의 파일과 날씨 파일만 있어도 됩니다.

- `asos_서울108_시간자료_2023_2026.csv` (모든 주제)
- `kpx_시간별_전국_전력수요량_2021_2025.csv` (주제 A)
- `서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv` (주제 B, 날짜·승하차마다 모든 역을 더한 파일)
- `에어코리아_서울_도시대기평균_시간별_2023_2026.csv` (주제 C)

출처: 기상청 기상자료개방포털, 한국전력거래소·서울교통공사(공공데이터포털), 한국환경공단 에어코리아. 공공누리 조건에 따라 출처를 밝힙니다.

> **[더 알아보기] ☆ 포털에서 직접 받아 합치기**
>
> 전력 수요 파일은 12장 미니 프로젝트와 같은 파일이고, 해마다의 파일을 잇는 방법도 12장과 같습니다. 지하철 원자료는 공공데이터포털에 해마다 올라와 있습니다. 열 이름이 `수송일자`, `승하차구분`, `13-14시간대` 꼴이고 합계 열이 없습니다. 늦은 시간 열은 해에 따라 다르고, 2023년은 기간이 겹치는 파일 두 개(1~9월, 1~12월)로 올라와 있습니다. 그래서 열 이름을 맞추고, 같은 날·같은 역·같은 승하차는 하나만 남긴 뒤 더합니다.

In [ ]:
# ☆ 포털에서 받은 역별 원자료를 서울 전체 하루 단위로 합치기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
RAW = sorted(glob.glob("서울교통공사_역별*승하차인원*.csv"))     # 받은 파일 이름에 맞게 고칩니다
if not RAW:
    print("포털에서 받은 파일이 없습니다. 강의 자료의 서울 전체 파일로 하십시오.")
else:
    parts = []
    for f in RAW:
        t = pd.read_csv(f, encoding="cp949")                    # 포털 파일의 한글은 cp949
        t = t.rename(columns={"수송일자": "날짜", "승하차구분": "구분"})
        t.columns = [c.replace("시간대", "시") for c in t.columns]   # 13-14시간대 → 13-14시
        late = [c for c in ["23-24시", "24시이후"] if c in t.columns]   # 늦은 시간 열은 해마다 다르다
        if late:
            t["23시이후"] = t[late].sum(axis=1)
            t = t.drop(columns=late)
        parts.append(t)
    raw = pd.concat(parts).drop_duplicates(["날짜", "역번호", "구분"], keep="last")  # 기간이 겹치는 파일
    hours = [c for c in raw.columns if c.endswith(("시", "이전", "이후"))]
    city = raw.groupby(["날짜", "구분"])[hours].sum()
    city["합계"] = city.sum(axis=1)
    city.reset_index().to_csv("지하철_직접_합친_파일.csv", index=False)
    print(len(RAW), "개 파일 →", city.shape, city.index[0][0], "~", city.index[-1][0])

**1단계: 데이터 신원 확인**(다섯 질문 ①, [파이썬 실습 13-1] 응용). 공통 날씨 파일을 하루 단위로 묶고, 고른 주제의 파일을 읽으시오. 기간, 빈칸, 한 행의 뜻, 이상한 값을 확인하시오.

In [ ]:
# 종합 프로젝트 공통: 기상청 서울(108) 시간자료 → 하루 단위
w = pd.read_csv("asos_서울108_시간자료_2023_2026.csv",
                usecols=["일시", "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)"])
w.columns = ["일시", "기온", "강수량", "풍속", "습도"]
w["일시"] = pd.to_datetime(w["일시"])
print("시간자료:", len(w), "행,", w["일시"].min(), "~", w["일시"].max())
print("빈칸:", w.isna().sum().to_dict())
w["강수량"] = w["강수량"].fillna(0)          # 비가 안 온 시각은 빈칸으로 적혀 있다
day = w.groupby(w["일시"].dt.normalize()).agg(기온=("기온", "mean"), 강수량=("강수량", "sum"),
                                             풍속=("풍속", "mean"), 습도=("습도", "mean"))
print("하루 단위:", day.shape)

강수량 빈칸 29,375개는 고장이 아니라 "비가 오지 않은 시각"입니다. 기상청은 비가 오지 않으면 칸을 비워 둡니다. 그래서 0으로 채웠습니다. 풍속의 빈칸 68개는 진짜 결측이라 그대로 두었습니다. 같은 빈칸이라도 열마다 뜻이 다릅니다(3장).

In [ ]:
# 주제 A: 전력 수요 (한국전력거래소, 전국, 시간별 MW)
pw = pd.read_csv("kpx_시간별_전국_전력수요량_2021_2025.csv", parse_dates=["날짜"])
hours = [f"{h}시" for h in range(1, 25)]
pw["GWh"] = pw[hours].mean(axis=1) * 24 / 1000        # 하루 전력량
A = pw.set_index("날짜")[["GWh"]].join(day, how="inner")
print(A.shape, A.index.min().date(), "~", A.index.max().date())
print(A.groupby(pd.cut(A["기온"], [-15, 0, 10, 20, 25, 35]), observed=True)["GWh"].mean().round(0))
A.plot.scatter(x="기온", y="GWh", s=6, color="#9a9a9a", figsize=(5, 3.2))
plt.xlabel("서울 하루 평균 기온 (°C)")
plt.ylabel("전국 하루 전력 수요 (GWh)")
plt.show()

In [ ]:
# 주제 B: 지하철 (서울교통공사, 모든 역을 더한 하루·시간대별 승하차)
sub = pd.read_csv("서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv", parse_dates=["날짜"])
B = sub[sub["구분"] == "승차"].set_index("날짜")[["13-14시", "합계"]] / 10000   # 만 명
month = B.groupby(B.index.to_period("M")).mean().round(1)
print(month.loc["2024-02":"2024-06"])
month.index = month.index.strftime("%y.%m")          # 23.01 꼴의 글자로
month["13-14시"].plot(marker="o", color="#C8234B", figsize=(6, 3), rot=0)
plt.ylabel("13~14시 승차 (만 명, 하루 평균)")
plt.xlabel("")
plt.show()

주제 B를 고른 사람은 여기서 멈춰야 합니다. 1장, 3장, 5장의 미니 프로젝트에서 만난 그 2024년 4월입니다. 13~14시 승차가 하루 평균 175.9만 명으로, 다른 달(20만 명 남짓)의 여덟 배쯤입니다. 그래서 그달의 하루 합계도 635.2만 명으로 부풀었습니다. 서울교통공사가 공개한 원자료에 그대로 있는 값입니다. 이 칸만 빼고 다시 더할지, 그달을 통째로 뺄지 정하고 그 결정을 기록해 두십시오. 3장은 이 칸을 앞뒤 시간대의 평균으로 채웠고, 5장은 그달을 통째로 뺐습니다.

In [ ]:
# 주제 C: 초미세먼지 (에어코리아 확정자료, 서울 도시대기 측정소 25곳 평균)
air = pd.read_csv("에어코리아_서울_도시대기평균_시간별_2023_2026.csv", parse_dates=["일시"])
air["날"] = (air["일시"] - pd.Timedelta(hours=1)).dt.normalize()   # 24시 기록은 그날 것
C = air.groupby("날")["PM25"].mean().to_frame("PM25").join(day, how="inner")
C["내일나쁨"] = (C["PM25"].shift(-1) > 35).astype(int)            # 36㎍/㎥ 이상이면 "나쁨"
C = C.iloc[:-1]
print(C.shape, " 다음 날이 '나쁨' 이상인 날:", round(C["내일나쁨"].mean() * 100, 1), "%")
print(C.groupby(C.index.month)["PM25"].mean().round(1).to_dict())

**2단계: 그림 한 장**(다섯 질문 ④, 4장). 질문에 답하는 그림 한 장을 그리시오. 그림 하나는 질문 하나에 답해야 합니다.

**3단계: 기준선 만들기**(다섯 질문 ②, 8·9·12장). 주제 A는 "지난주 같은 요일", 주제 C는 "내일도 오늘과 같다"를 기준선으로 쓰시오. 주제 B는 같은 달, 같은 요일의 보통 날과 견주시오. 날씨가 아니라 요일과 계절이 만든 차이를 빼기 위해서입니다.

**4단계: 이 책의 기법 쓰기**(다섯 질문 ③). A는 기온과 휴일로 회귀를(8장), B는 비 온 날과 아닌 날의 차이를 검정으로(5장), C는 로지스틱 회귀나 랜덤 포레스트로 분류를(9·10장) 하시오. A와 C는 2023~2024년으로 배우고 그 뒤로 시험하시오. C의 문턱값은 시험 기간이 아니라 그 앞의 검증 기간으로 고르시오([파이썬 실습 13-9]).

**5단계: 해석과 한계**(다섯 질문 ⑤). [표 13-2]의 다섯 질문에 한 줄씩 답하는 표를 만드시오. 예를 들어 주제 A에서 서울 기온 하나로 전국 수요를 맞히는 것은 무엇을 놓칠까요? 그날의 실제 기온은 전날 밤에는 알 수 없고 예보만 알 수 있다는 점은 어떻게 다룰까요?

**3시간 계획.** 첫 1시간에 1~2단계, 다음 1시간에 3~4단계, 마지막 1시간에 5단계와 제출물 정리를 합니다.

**제출물.** 그림 2장(2단계 그림, 기준선과 모델을 견준 그림), 숫자 표 1개(기준선과 모델의 점수), 결론 5문장 이내, 다섯 질문 표.

**[표 13-4] 종합 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 기간, 빈칸의 뜻, 이상한 값(예: 2024년 4월)을 찾아 처리하고 기록함 | 빈칸만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 데이터의 주기에 맞는 기준선과 시간 순서로 견줌 | 기준선은 있으나 무작위로 나눔 | 기준선 없이 점수만 씀 |
| 기법 적용(30%) | 앞 장의 기법을 맞게 쓰고, 문턱값이나 비교 조건을 시험 기간 밖에서 정함 | 기법은 맞으나 시험 기간으로 고름 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 다섯 질문에 모두 답하고, 모델이 기준선을 못 이긴 경우도 그대로 씀 | 한계를 한두 가지만 씀 | 숫자만 나열함 |

### 내 분석

여기부터 고른 주제의 2~5단계를 직접 해 보십시오.

In [ ]:
# 2단계: 질문에 답하는 그림 한 장

In [ ]:
# 3단계: 기준선 만들기

In [ ]:
# 4단계: 이 책의 기법 쓰기 (시간 순서로 나누어 시험하기)

**5단계: 해석과 한계** — 다섯 질문에 한 줄씩 답해 보십시오.

| 질문 | 내 답 |
|---|---|
| ① 누가, 왜, 어떻게 모았나? | |
| ② 가장 단순한 답과 견주었나? | |
| ③ 실제로 쓰일 때처럼 시험했나? | |
| ④ 틀리면 무엇을 잃나? | |
| ⑤ 아직 모르는 것은 무엇인가? | |

## ✏️ [연습문제 13-6] 공구 마모 고장을 규칙으로 찾기

AI4I 데이터에서 공구 마모 고장(TWF)이 일어난 작업의 공구 마모 시간 범위를 구하고, "마모 시간 200분 이상이면 고장 경보"라는 규칙의 정밀도와 재현율을 구하시오. 결과를 보고 이 규칙을 공장에서 쓸지 판단하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오. [파이썬 실습 13-6]의 `df`를 이어서 씁니다.

In [ ]:
twf = df["TWF"] == 1
print("공구 마모 고장(TWF):", twf.sum(), "건, 마모 시간",
      df.loc[twf, "Tool wear [min]"].min(), "~", df.loc[twf, "Tool wear [min]"].max(), "분")
rule = df["Tool wear [min]"] >= 200
print("마모 200분 이상:", rule.sum(), "건, 그 가운데 TWF:", (rule & twf).sum(), "건")
print("정밀도:", round((rule & twf).sum() / rule.sum() * 100, 1), "%, 재현율:",
      round((rule & twf).sum() / twf.sum() * 100, 1), "%")